## Modeling approach

In this notebook, we explore modeling approach with XGBoost. Our dataset has the following structure:

| FSA | features | concentration |
| :--- | :---: | ---: |
| A | features(A) | $r_{A,1}$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| A | features(A) | $r_{A,n_A}$ |
| B | features(B) | $r_{B,1}$ |
| $\vdots$ | $\vdots$| $\vdots$ |
| B | features(B) | $r_{B,n_B}$ |
| $\vdots$ | $\vdots$ | $\vdots$ |

Here features(A) denotes all the features columns of that particular FSA (which remains the same per FSA), and $r_{A, i}$ denotes the radon concentration level of $i$-th measurement in the FSA A. We add a new column names `y_binary`, which is equal to 1, if concentration >200, else 0. We use XGBoost classifier to get the predicted probabilities `y_pred` for of class 1 (using `.predict_prob()`). For each FSA, we use the proportion of measurements above 200 divided by the total number of measurements in that FSA as `y_true`. We use error metrics such as MAE, Weighted MAE (w.r.t the number of measurements per FSA) etc to evaluate the quality of our predicted values `y_pred` againsts our aggregated true values `y_true`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import sys
# ---------------------------------------------------------------------
# Add project root to Python path
# ---------------------------------------------------------------------


# project root = two levels above notebooks
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
from src.data.data_loading import (
    load_full_dataset,
    load_full_training_pool,
    load_test_data,
    load_training_data,
    load_validation_data
)
train_df = load_training_data(cv_fold=0).drop(columns= ['spatial_cluster', 'is_test', 'cv_fold'])
val_df = load_validation_data(cv_fold=0).drop(columns= ['spatial_cluster', 'is_test', 'cv_fold'])
import xgboost as xgb

In [ ]:
from src.modeling.baseline import baseline_proportion
baseline_df = baseline_proportion()
print(f"Number of FSAs with non-zero proportion of measurements exceeding 200"
      f": {len(baseline_df[baseline_df['proportion'] != 0])/ len(baseline_df)}")

In [ ]:
print(f"Length of train: {len(train_df)}")
print(f"Length of train: {len(val_df)}")
print(f"Unique FSAs in train: {train_df['FSA'].nunique()}")
print(f"Unique FSAs in val: {val_df['FSA'].nunique()}")

In [ ]:
## Helper Functions
from src.modeling.utils import add_y_binary, fsa_grouped_features_and_proportion_above200, no_of_measurements_per_fsa

def extract_X_y(df, is_train, approach = "Naive",):
    if is_train == True:
        y = df['y_binary']
    else:
        df = fsa_grouped_features_and_proportion_above200(df)
        y = df['y_mean']

    if approach == "Naive":
        X = df.drop(columns=['concentration', 'provinceterritory', 'FSA', 'geometry', 'y_binary', 'y_mean'], 
                    errors = 'ignore')
    elif approach == "highly_correlated_only":
        columns_to_keep = ['hous_frac_type_single_detached', 'mean_uranium', 'geolprov_interior_platform',
                           'socioeco_frac_housing_burden', 'hous_median_value', 'hous_frac_type_other_attached']
        columns_to_drop = df.columns.difference(columns_to_keep)
        X = df.drop(columns= columns_to_drop)
        
    return (X,y)


In [ ]:
from sklearn.metrics import brier_score_loss, log_loss, mean_absolute_error
pos_count = add_y_binary(train_df)['y_binary'].sum()
neg_count = len(train_df) - pos_count

max_depths = [1, 2, 3, 5, 10]
learning_rates = [0.05, 0.1, 0.5]
#max_depths = [2]
#learning_rates = [0.05]
maes = {}
wmaes  = {}

for max_depth in max_depths:
    for lr in learning_rates:
        clf = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            scale_pos_weight=neg_count / pos_count,
            max_depth= max_depth,
            n_estimators=300,
            learning_rate= lr ,
            random_state=42
        )
        X, y = extract_X_y(train_df, is_train= True)
        clf.fit(
            X,
            y,
            verbose = False
            )
        
        X_val, y_true = extract_X_y(val_df, is_train = False)
        y_pred_prob = clf.predict_proba(X_val)[:,1]
        
        no_of_measurements = no_of_measurements_per_fsa(val_df)['no_of_measurements'].tolist()
        #bsl = brier_score_loss(y_true= np.ones(len(X_val)), y_proba=y_pred_prob)
        #logloss = log_loss(y_true= np.ones(len(X_val)), y_pred=y_pred_prob)
        mae = mean_absolute_error(y_true=y_true, y_pred=y_pred_prob)
        wmae = mean_absolute_error(y_true=y_true, y_pred=y_pred_prob, sample_weight=no_of_measurements)

        ## save error scores
        maes[f"({max_depth},{lr})"] = mae
        wmaes[f"({max_depth},{lr})"] = wmae

        print(f"Iteration cycle with max_depth {max_depth} and learning rate {lr}:")
        #print(f"Brier score: {bsl:.2f}.")
        #print(f"Log loss score with max_depth {max_depth} and learning rate {lr}: {logloss:.2f}.")
        print(f"MAE : {mae:.2f}.")
        print(f"Weighted MAE (weighted by number of measurements per FSA): {wmae:.2f}.")


In [ ]:
print("Best parameters when the models used all the features:")
print(f"Best parameters in terms of MAE: {min(maes, key = maes.get)}. MAE with these parameters: {maes[min(maes, key = maes.get)]:.2f}.")
print(f"Best parameters in terms of Weighter MAE: {min(wmaes, key = wmaes.get)}. MAE with these parameters: {wmaes[min(wmaes, key = wmaes.get)]:.2f}.")

In [ ]:
max_depths = [1, 2, 3, 5, 10]
learning_rates = [0.05, 0.1, 0.5]
#max_depths = [2]
#learning_rates = [0.05]
maes = {}
wmaes  = {}

for max_depth in max_depths:
    for lr in learning_rates:
        clf = xgb.XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            scale_pos_weight=neg_count / pos_count,
            max_depth= max_depth,
            n_estimators=300,
            learning_rate= lr ,
            random_state=42
        )
        X, y = extract_X_y(train_df, is_train= True, approach= 'highly_correlated_only')
        clf.fit(
            X,
            y,
            verbose = False
            )
        
        X_val, y_true = extract_X_y(val_df, is_train = False, approach='highly_correlated_only')
        y_pred_prob = clf.predict_proba(X_val)[:,1]
        
        no_of_measurements = no_of_measurements_per_fsa(val_df)['no_of_measurements'].tolist()
        #bsl = brier_score_loss(y_true= np.ones(len(X_val)), y_proba=y_pred_prob)
        #logloss = log_loss(y_true= np.ones(len(X_val)), y_pred=y_pred_prob)
        mae = mean_absolute_error(y_true=y_true, y_pred=y_pred_prob)
        wmae = mean_absolute_error(y_true=y_true, y_pred=y_pred_prob, sample_weight=no_of_measurements)

        ## save error scores
        maes[f"({max_depth},{lr})"] = mae
        wmaes[f"({max_depth},{lr})"] = wmae

        print(f"Iteration cycle with max_depth {max_depth} and learning rate {lr}:")
        #print(f"Brier score: {bsl:.2f}.")
        #print(f"Log loss score with max_depth {max_depth} and learning rate {lr}: {logloss:.2f}.")
        print(f"MAE : {mae:.2f}.")
        print(f"Weighted MAE (weighted by number of measurements per FSA): {wmae:.2f}.")
        print("\n")

In [ ]:
print("Best parameters when we use only selected features (total 6 features):")
print(f"Best parameters in terms of MAE: {min(maes, key = maes.get)}. MAE with these parameters: {maes[min(maes, key = maes.get)]:.2f}.")
print(f"Best parameters in terms of Weighter MAE: {min(wmaes, key = wmaes.get)}. MAE with these parameters: {wmaes[min(wmaes, key = wmaes.get)]:.2f}.")